# Stage 2 Notebook 14 - Exp2I CLRKD + OHEM + stronger prior encoder

Exp2H (NB13) was a lateral move on classification: best_f1 spiked to 0.245 at epoch 1-2 (vs Exp2G's 0.168) but collapsed back to 0.083 by epoch 10, while pos_score - neg_score dropped from +0.122 to +0.016. cls_pos rose 0.083 -> 0.133 and pos_score_mean fell 0.633 -> 0.580 -- positive predictions actively degrade as training progresses.

Diagnosis: dynamic-k matching makes positive vs negative a soft boundary on geometrically-similar priors. Geometry losses train per_lane to encode 'what curve goes through here' (similar across priors near the same GT lane), so cls struggles to learn 'is *this* prior matched'. 1500 negative priors' gradients dilute the 40 positives' representational pull through the shared per_lane.

Exp2I targets that mechanism with two surgical changes:

- **OHEM hard-negative mining**. Per image, keep only top-K hardest negatives where K = max(32, 4 * num_pos). Reduces negative gradient count from ~1500 to ~160 per batch, so positive signal can dominate the per_lane representation.
- **Stronger prior embedding encoder**. Replace the raw 3-d prior concat with `Linear(3, 64) + ReLU` before concat with per_lane. At init the prior contributes ~33% of cls_head input instead of ~2%; the cls head can rely more heavily on prior position as a discriminator.

Other tweaks based on Exp2H telemetry: `w_cls: 6.0 -> 4.0` and `w_iou: 1.0 -> 1.5`. Exp2H's geometry slightly regressed (matched_iou 0.43 -> 0.40); OHEM is the better lever for cls so we don't need w_cls=6, and we restore some geometry weight.

Backbone (RMT + GCA), detection head (DETR), CLRKDLaneHead architecture (3 stages, 36 sample points, dynamic-k matching), ASL with gamma_pos=0/gamma_neg=4 are unchanged from Exp2H.

Reference: Shrivastava et al. 2016 'Training Region-based Object Detectors with Online Hard Example Mining'.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [4]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [5]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: tiny forward + backward through OHEM + prior_embed_encoder + ASL.
# Must print 'OK exp09_*.yaml' with shapes before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_smoke.log
OK exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.5616 det_loss=3.7426 grad_cos=0.0908 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49771663546562195, 'gate/lane_mean': 0.5025568604469299, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [6]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp09_rmt_gca_clrkd_ohem_prior_encoder_joint.yaml --curve-ta

0

## What to watch in Exp2I training

Reference epoch 10:
- Exp2G: point_mae=0.3244, matched_iou=0.4285, best_f1=0.075, pos-neg=0.004.
- Exp2H: point_mae=0.3285, matched_iou=0.4018, best_f1=0.083, pos-neg=0.016.

Strong signals that OHEM + prior encoder worked:

- `val/lane_exist_best_f1` stays above 0.20 from epoch 1, rises to >= 0.40 by epoch 5, >= 0.65 by epoch 10. The key test is whether the epoch-1 signal persists this time (Exp2H peaked at 0.245 then collapsed).
- `val/lane_exist_pos_score_mean - val/lane_exist_neg_score_mean` >= 0.15 at epoch 10 and does NOT decay over training. Exp2H went +0.122 -> +0.016 (collapse); Exp2I should go upward or hold flat.
- `val/lane/cls_pos` strictly DECREASES over training (Exp2H regressed 0.083 -> 0.133). With OHEM, pos gradient should not be diluted, so pos predictions should improve, not degrade.
- `val/lane/cls_neg` decreases. With only top-K hardest negatives in the loss, the cls_neg value reported is now the mean of the hardest negatives -- a different distribution. Expect cls_neg around 0.05-0.10 (higher than Exp2H's 0.04 because we're reporting hard cases, not the easy mean).
- `pred_lanes / batch` drops below 500 (Exp2G/H were stuck at ~1500 = all priors above thr=0.3).
- **Geometry holds**: `val/lane_point_mae <= 0.34` and `val/matched_line_iou >= 0.30` at epoch 10. With w_iou raised back to 1.5 we expect matched_iou closer to Exp2G's 0.43 than Exp2H's 0.40.

Failure signals:

- best_f1 still below 0.20 at epoch 10 -> the per_lane representation itself is the bottleneck, not the loss formulation. Next step would be Option B: separate cls feature pathway with its own gradient, or replace the existence task with a direct LineIoU regression target.
- Geometry regresses sharply (point_mae > 0.36) -> w_cls=4 is still too high relative to w_iou=1.5. Pull w_cls to 2.5.
- pred_lanes drops to ~0 with recall collapse -> OHEM is too aggressive; raise `cls_ohem_topk_per_pos` from 4 to 8 or set `cls_ohem_min_topk: 64`.

After short10, run Notebook 08 (which now includes exp09 candidates) to plot Exp2G / Exp2H / Exp2I side-by-side.